## Bronze Layer (Deliverable 1.1.1 and 1.1.2)

This notebook completes the Bronze layer of the Medallion Architecture for the pump sensor dataset. It focuses on raw data ingestion, structural validation, and standardization so that the rest of the analytical workflow has a clean and consistent foundation.

The key goals of this notebook are:

- To ingest the raw pump sensor data from the source files.
- To validate the structure, column consistency, data types, and timestamp integrity.
- To standardize the dataset into the unified Bronze schema, including meta fields, identity fields, and provenance tracking.
- To export the Bronze dataset in a reproducible format that can be used by the Silver Pre-EDA and Silver EDA notebooks.

Outputs from this notebook support the project write-up in Section B and Section C by:

- Establishing the standardized dataset required for all downstream analysis and modeling described in Section C.2 and C.2.A.
- Providing the meta-data, event identifiers, timestamp fields, and structural consistency checks required for later segmentation of normal vs abnormal periods used in Section C.4.
- Ensuring that the Silver and Gold layers receive a clean, well-formed dataset, enabling reliable feature behavior analysis, anomaly profiling, and model comparison.
- Serving as the reproducible starting point for the entire analytical pipeline, consistent with the Medallion Architecture described in B.3.

## Overview

This notebook ingests the raw pump dataset, validates the Bronze structure, and exports a reproducible Bronze foundation for Silver processing. It is part of the Bronze portion of the capstone workflow and should be read as a notebook-first implementation step rather than a separate pipeline redesign.


## Objectives

- Document how this notebook ingests the raw pump dataset, validates the Bronze structure, and exports a reproducible Bronze foundation for Silver processing.
- Preserve the existing notebook-first execution flow, configuration usage, logger behavior, ledger behavior, and artifact outputs.
- Make the notebook easier to review by separating setup, validation, processing, outputs, and handoff context.
- Clarify how the Bronze output becomes the stable input foundation for Silver processing.


## Code Reference

A detailed code-cell reference for this notebook is maintained in the project documentation:`docs/technical_reference/notebook_code_references/EDA_Notebook_Pump_Bronze_01_Preprocessing_code_reference.md`


## Environment Setup and Utility Imports

In this section I am loading the libraries and project utilities needed for the Bronze preprocessing stage. The point here is to get the notebook ready to run using the project’s existing utility structure instead of creating another one-off pattern.

This includes the shared config loader, path utilities, logging helpers, artifact utilities, truth/lineage helpers, file I/O helpers, profiling helpers, database helpers, and Weights & Biases tracking helpers. These imports support the rest of the notebook, but they should not change how the project is already organized.

In [1]:
import os
import glob
from pathlib import Path
import yaml
import sys

import json
import logging
import wandb
from datetime import datetime, timezone

from typing import Optional, Tuple, List, Any, Dict, Mapping, cast

import pandas as pd 
import numpy as np

import hashlib

# Custom Utilities Module
from utils.core.paths import get_paths
from utils.core.file_io import ingest_data, save_data, load_json
from utils.core.logging_setup import (
    configure_logging, 
    log_layer_paths
)

from utils.core.logging_profiler import profile_dataframe

from utils.core.truths import (
    make_process_run_id,
    build_file_fingerprint,
    extract_truth_hash,
    identify_meta_columns,
    identify_feature_columns,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record,
    find_truth_record_by_hash, 
    load_truth_record_by_hash, 
    load_parent_truth_record_from_dataframe,
)

'''
from utils.core.wandb_utils import (
    log_metrics,
    log_dataframe_head,
    log_dir_as_artifact,
    log_files_as_artifact,
)
'''


from utils.core.wandb_utils import finalize_wandb_stage

from utils.core.config_loader import (
    load_pipeline_config,
    build_truth_config_block,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.database.postgres import (
    get_engine_from_env, 
    read_sql_dataframe,
)
from utils.database.layer_postgres import (
    read_layer_dataframe, 
    write_layer_dataframe, 
    prepare_layer_dataframe,
)

from utils.database.sql_notebook_helpers import (
    delete_dataset_run_rows,
    execute_many,
    get_existing_dataframe,
    get_row_value,
    log_data_quality_event,
    log_pipeline_artifact,
    preview_sql_table,
    row_to_payload,
    sql_table_ref,
    to_builtin,
    to_json_string,
    to_scalar,
    upsert_pipeline_run,
)


from utils.database.medallion_sql_writers import (
    log_gold_05_anomaly_detection_summary_sql,
    log_silver_eda_sql,
    write_bronze_sensor_observations_sql,
    write_gold_baseline_scores_sql,
    write_gold_cascade_scores_sql,
    write_gold_model_comparison_results_sql,
    write_gold_preprocessed_features_sql,
    write_silver_sensor_observation_features_sql,
)


from utils.core.artifacts import (
    build_artifact_dirs,
    build_artifact_dirs_from_config,
    artifact_file_path,
)

from utils.core.notebook_context import load_notebook_context


# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)



#### Define configuration mapping guards

This cell defines helper logic used by the surrounding `Environment Setup and Utility Imports` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [2]:
def cfg_require_mapping(value: object, name: str) -> Mapping[str, Any]:
    if not isinstance(value, Mapping):
        raise TypeError(
            f"{name} must be a mapping, got {type(value).__name__}: {value!r}"
        )

    return cast(Mapping[str, Any], value)


def cfg_optional_mapping(value: object | None, name: str) -> Mapping[str, Any]:
    if value is None:
        return {}

    return cfg_require_mapping(value, name)

----

## Load Project Paths, Configuration, and Runtime Settings

Here I am loading the resolved project paths and the Bronze configuration settings that control how this notebook runs. This is where the notebook gets its project root, artifact root, data locations, config values, runtime options, and source-mode settings.

The important thing in this section is keeping the standards stable. `paths = get_paths()` stays as the paths object from `paths.py`, while config-resolved paths stay separate. I do not want `PATHS` switching back and forth between an object and a dictionary because that is what caused the `PATHS.artifacts` issue.

This keeps Bronze aligned with the method I have been using so far instead of changing the notebook again when it is already close to complete.

In [3]:
# Shared notebook context
CONTEXT_STAGE = "bronze"
CONTEXT_DATASET = "pump"
CONTEXT_LAYER = "bronze"
CONFIG_RUN_MODE = "train"
CONFIG_PROFILE = "default"
CONTEXT_LOG_FILE = "bronze.log"

CTX = load_notebook_context(
    stage=CONTEXT_STAGE,
    dataset=CONTEXT_DATASET,
    mode=CONFIG_RUN_MODE,
    profile=CONFIG_PROFILE,
    logger_child_name="capstone.bronze",
    log_filename=CONTEXT_LOG_FILE,
)

# Shared aliases used throughout the notebook
paths = CTX.paths
CONFIG = CTX.config
CONFIG_MAP = CTX.config
STAGE_CFG = CTX.stage_config
BRONZE_CFG = CTX.stage_config
RESOLVED_PATHS = CTX.resolved_paths
FILENAMES = CTX.filenames
VERSIONS_CFG = CTX.versions
RUNTIME_CFG = CTX.runtime
DATASET_CFG = CTX.dataset_config
WANDB_CFG = CTX.wandb
EXECUTION_CFG = CTX.execution
PIPELINE = CTX.pipeline
logger = CTX.logger
ledger = CTX.ledger
LOG_PATH = CTX.log_path
CONTEXT_RECIPE_ID = CTX.recipe_id

logger.info(
    "Notebook context loaded",
    extra={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
    },
)

ledger.add(
    kind="step",
    step="context_loaded",
    message="Loaded shared notebook context.",
    data={
        "stage": CONTEXT_STAGE,
        "recipe_id": CONTEXT_RECIPE_ID,
        "dataset": CONTEXT_DATASET,
        "mode": CONFIG_RUN_MODE,
        "profile": CONFIG_PROFILE,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)

2026-06-14 20:35:50,098 | INFO | capstone.bronze | bronze stage starting
2026-06-14 20:35:50,100 | INFO | capstone.bronze | LEDGER | {'ts_utc': '2026-06-14T20:35:50.100000+00:00', 'stage': 'bronze', 'recipe': 'bronze__v001', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger from shared notebook context', 'why': None, 'consequence': None, 'data': {'stage': 'bronze', 'recipe_id': 'bronze__v001', 'dataset': 'pump', 'mode': 'train', 'profile': 'default', 'log_path': '/workspace/logs/bronze.log'}}
2026-06-14 20:35:50,102 | INFO | capstone.bronze | Notebook context loaded
2026-06-14 20:35:50,103 | INFO | capstone.bronze | LEDGER | {'ts_utc': '2026-06-14T20:35:50.103465+00:00', 'stage': 'bronze', 'recipe': 'bronze__v001', 'kind': 'step', 'step': 'context_loaded', 'message': 'Loaded shared notebook context.', 'why': None, 'consequence': None, 'data': {'stage': 'bronze', 'recipe_id': 'bronze__v001', 'dataset': 'pump', 'mode': 'train', 'profile': 'default', 'log_path': '/workspace/l

{'ts_utc': '2026-06-14T20:35:50.103465+00:00',
 'stage': 'bronze',
 'recipe': 'bronze__v001',
 'kind': 'step',
 'step': 'context_loaded',
 'message': 'Loaded shared notebook context.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'bronze',
  'recipe_id': 'bronze__v001',
  'dataset': 'pump',
  'mode': 'train',
  'profile': 'default',
  'log_path': '/workspace/logs/bronze.log'}}

### Notebook Level Configuration

In [4]:
TRUTH_CONFIG = cast(Dict[str, Any], build_truth_config_block(CONFIG))
TRUTH_CONFIG["pipeline"] = PIPELINE

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Stage details ----

STAGE = "bronze"

LAYER_NAME = str(BRONZE_CFG["layer_name"])

BRONZE_VERSION = str(VERSIONS_CFG["bronze"])
TRUTH_VERSION = str(VERSIONS_CFG["truth"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

PIPELINE_MODE = str(PIPELINE["execution_mode"])
RUN_MODE = str(RUNTIME_CFG.get("mode", CONFIG_RUN_MODE))
CONFIG_PROFILE = str(RUNTIME_CFG.get("profile", CONFIG_PROFILE))

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Dataset details ----

DATASET_NAME_ARGUMENT = None

DATASET_NAME_CONFIG = str(DATASET_CFG.get("name", "pump"))
DATASET_NAME = DATASET_NAME_CONFIG.strip().lower()

DATASET_CANDIDATES = list(BRONZE_CFG["dataset_candidates"])

SPLIT_STATUS = str(DATASET_CFG["split_status"])
LABEL_TYPE = DATASET_CFG.get("label_type")
LABEL_SOURCE = DATASET_CFG.get("label_source")

RUN_ID = str(DATASET_CFG["run_id"])
ASSET_ID = str(DATASET_CFG["asset_id"])

# DataFrame-friendly values.
LABEL_TYPE_DF = pd.NA if LABEL_TYPE is None else LABEL_TYPE
LABEL_SOURCE_DF = pd.NA if LABEL_SOURCE is None else LABEL_SOURCE

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Processing lineage ----

PROCESS_RUN_ID = make_process_run_id(
    str(BRONZE_CFG["process_run_id_prefix"])
)

ADD_RECORD_ID = bool(BRONZE_CFG["add_record_id"])
RECORD_ID_INPUTS = list(BRONZE_CFG["record_id_inputs"])
RECORD_ID_METHOD = str(BRONZE_CFG["record_id_method"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- W&B ----

WANDB_PROJECT = str(WANDB_CFG.get("project", "capstone"))
WANDB_ENTITY = str(WANDB_CFG.get("entity", ""))
WANDB_RUN_NAME = f"{BRONZE_VERSION}"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- File names ----

BRONZE_TRAIN_DATA_FILE_NAME = str(FILENAMES["bronze_train_file_name"])

RAW_FILE_NAME = str(DATASET_CFG["raw_file_name"])

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Base paths only ----

ARTIFACTS_ROOT = Path(str(RESOLVED_PATHS["artifacts_root"]))

RAW_DATA_PATH = Path(str(RESOLVED_PATHS["data_raw_dir"]))
RAW_FILE_PATH = Path(str(RESOLVED_PATHS["raw_file_path"])).parent

BRONZE_DATA_PATH = Path(str(RESOLVED_PATHS["data_bronze_train_dir"]))

TRUTHS_PATH = Path(str(RESOLVED_PATHS["truths_dir"]))
TRUTH_INDEX_PATH = Path(str(RESOLVED_PATHS["truth_index_path"]))
LOGS_PATH = Path(str(RESOLVED_PATHS["logs_root"]))

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Source mode ----

BRONZE_SOURCE_MODE = "file"  # "file" | "postgres_handoff"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# ---- Postgres handoff defaults ----

DATASET_NAME_POSTGRES = DATASET_NAME

POSTGRES_SOURCE_TABLE_NAME = "bronze_observations_input_stage"

POSTGRES_SOURCE_TABLE_DATASET_MAP = {
    POSTGRES_SOURCE_TABLE_NAME: str(DATASET_NAME_POSTGRES).strip(),
}

LAYER_SCHEMA = "bronze"

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# W&B

WANDB_DIR = set_wandb_dir_from_config(CONFIG)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####
# Path failsafes

BRONZE_DATA_PATH.mkdir(parents=True, exist_ok=True)
TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### ####

print("Project root:", paths.root)
print("Artifacts root:", ARTIFACTS_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("Bronze data path:", BRONZE_DATA_PATH)
print("Bronze source mode:", BRONZE_SOURCE_MODE)
print("Configured dataset name:", DATASET_NAME_CONFIG)

Project root: /workspace
Artifacts root: /workspace/artifacts
Raw data path: /workspace/data/raw
Bronze data path: /workspace/data/bronze/train
Bronze source mode: file
Configured dataset name: pump


----

## Defer Bronze Artifact Folder Creation Until the Dataset Name Is Resolved

This cell intentionally does not create the final Bronze artifact folders yet. Earlier versions were creating artifact folders from a `PROVISIONAL_DATASET_NAME` and then resolving the dataset name later, which made the notebook harder to follow and risked duplicate or incorrectly named artifact folders.

The better method is to let the artifact util create the folders once, after ingestion confirms the final dataset identity. That keeps the artifact folder creation clean, traceable, and consistent with the actual Bronze output.

So at this step, we are only preparing for artifact creation. The real Bronze artifact directories are created later with `build_artifact_dirs()` after `RESOLVED_DATASET_NAME` is available.

In [6]:
BRONZE_ARTIFACT_DIRS = {}

print("Bronze artifact directory creation deferred until after dataset resolution.")


Bronze artifact directory creation deferred until after dataset resolution.


----

## SQL Runtime Context

This step establishes the Postgres connection and resolves the dataset/run identifiers used for SQL logging and medallion table writes.

The purpose is not to relabel Bronze as a synthetic generator run. Bronze can still write to Postgres using dataset and run metadata, but the notebook itself should stay focused on Bronze preprocessing. The SQL identifiers are for traceability and database writes; they should not drive the final artifact folder naming unless that matches the resolved Bronze dataset identity.


In [7]:

engine = get_engine_from_env()

CAPSTONE_SCHEMA: str = str(os.getenv("CAPSTONE_SCHEMA", "capstone"))


def first_non_empty_string(*values: object) -> str | None:
    """
    Return the first non-empty string-like value from a list of candidates.

    This helper skips None, empty strings, whitespace-only strings, and
    dictionaries. It is used to resolve dataset/run identifiers without
    accidentally accepting an invalid config object or a placeholder None value.
    """
    for value in values:
        if value is None:
            continue

        if isinstance(value, dict):
            continue

        text_value = str(value).strip()

        if text_value:
            return text_value

    return None


dataset_config = (
    cast(Dict[str, Any], CONFIG.get("dataset", {}))
    if isinstance(CONFIG.get("dataset", {}), dict)
    else {}
)

dataset_config_name = dataset_config.get("name")
dataset_config_id = dataset_config.get("dataset_id", dataset_config.get("id"))
dataset_config_run_id = dataset_config.get("run_id")
dataset_config_asset_id = dataset_config.get("asset_id")

is_synthetic_run = str(RUN_MODE).lower() in {
    "synthetic",
    "synthetic_train",
    "synthetic_run",
    "simulation",
}

if is_synthetic_run:
    raw_dataset_id = first_non_empty_string(
        os.getenv("CAPSTONE_DATASET_ID"),
        os.getenv("SYNTHETIC_DATASET_ID"),
        globals().get("DATASET_ID"),
        dataset_config_id,
        globals().get("DATASET_NAME_CONFIG"),
        dataset_config_name,
        "pump_synthetic_v1",
    )

    raw_run_id = first_non_empty_string(
        os.getenv("CAPSTONE_RUN_ID"),
        os.getenv("SYNTHETIC_RUN_ID"),
        globals().get("DATASET_RUN_ID"),
        dataset_config_run_id,
        globals().get("RUN_ID"),
        globals().get("RUN_ID_DEFAULT_FALLBACK"),
        "synthetic_run_001",
    )

    raw_asset_id = first_non_empty_string(
        os.getenv("CAPSTONE_ASSET_ID"),
        os.getenv("SYNTHETIC_ASSET_ID"),
        globals().get("ASSET_ID"),
        dataset_config_asset_id,
        globals().get("ASSET_ID_DEFAULT_FALLBACK"),
        "pump_asset_001",
    )

else:
    raw_dataset_id = first_non_empty_string(
        os.getenv("CAPSTONE_DATASET_ID"),
        globals().get("DATASET_ID"),
        dataset_config_id,
        globals().get("DATASET_NAME_CONFIG"),
        dataset_config_name,
        "pump",
    )

    raw_run_id = first_non_empty_string(
        os.getenv("CAPSTONE_RUN_ID"),
        globals().get("DATASET_RUN_ID"),
        dataset_config_run_id,
        globals().get("RUN_ID"),
        globals().get("RUN_ID_DEFAULT_FALLBACK"),
        "run__001",
    )

    raw_asset_id = first_non_empty_string(
        os.getenv("CAPSTONE_ASSET_ID"),
        globals().get("ASSET_ID"),
        dataset_config_asset_id,
        globals().get("ASSET_ID_DEFAULT_FALLBACK"),
        "asset__001",
    )


if raw_dataset_id is None:
    raise ValueError(
        "DATASET_ID could not be resolved. "
        "Set CAPSTONE_DATASET_ID or configure CONFIG['dataset']['name'] / "
        "CONFIG['dataset']['dataset_id']."
    )

if raw_run_id is None:
    raise ValueError(
        "RUN_ID could not be resolved. "
        "Set CAPSTONE_RUN_ID, CONFIG['dataset']['run_id'], or default_fallbacks.run_id."
    )

if raw_asset_id is None:
    raise ValueError(
        "ASSET_ID could not be resolved. "
        "Set CAPSTONE_ASSET_ID, CONFIG['dataset']['asset_id'], or default_fallbacks.asset_id."
    )


DATASET_ID: str = raw_dataset_id
RUN_ID: str = raw_run_id
ASSET_ID: str = raw_asset_id


print(f"SQL schema: {CAPSTONE_SCHEMA}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Run ID: {RUN_ID}")
print(f"Asset ID: {ASSET_ID}")
print(f"Synthetic run: {is_synthetic_run}")

SQL schema: capstone
Dataset ID: pump
Run ID: run__001
Asset ID: asset__001
Synthetic run: False


In [ ]:
required_context_vars = [
    "CTX",
    "paths",
    "CONFIG",
    "CONFIG_MAP",
    "STAGE_CFG",
    "RESOLVED_PATHS",
    "FILENAMES",
    "VERSIONS_CFG",
    "RUNTIME_CFG",
    "DATASET_CFG",
    "WANDB_CFG",
    "EXECUTION_CFG",
    "PIPELINE",
    "logger",
    "ledger",
    "LOG_PATH",
]

missing_context_vars = [name for name in required_context_vars if name not in globals()]
if missing_context_vars:
    raise NameError(f"Missing required shared context variables: {missing_context_vars}")

logger.info(
    "Context sanity check passed",
    extra={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
)

ledger.add(
    kind="check",
    step="context_sanity_check",
    message="Verified shared notebook context variables are available.",
    data={
        "stage": CTX.stage,
        "recipe_id": CTX.recipe_id,
        "dataset": CTX.dataset,
        "mode": CTX.mode,
        "profile": CTX.profile,
        "log_path": str(CTX.log_path),
    },
    logger=logger,
)

pd.DataFrame(
    [
        {"name": name, "status": "present"}
        for name in required_context_vars
    ]
)

In [ ]:
bronze_required_context_vars = [
    "BRONZE_CFG",
    "DEFAULT_FALLBACKS",
]

missing_bronze_context_vars = [
    name for name in bronze_required_context_vars
    if name not in globals()
]

if missing_bronze_context_vars:
    raise NameError(f"Missing Bronze context variables: {missing_bronze_context_vars}")

logger.info("Bronze context sanity check passed")

## SQL Smoke Check

Before writing anything to Postgres, I run a quick smoke check to confirm the database connection works and the expected schemas/tables exist. This is a lightweight safety check so I can catch environment or schema problems early instead of finding them after the Bronze dataframe is already processed.

This helps confirm that the Docker/Postgres side of the capstone is available and that the notebook can communicate with the database layer before it reaches the final SQL write step.

In [8]:

sql_smoke_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        table_schema,
        table_name
    FROM information_schema.tables
    WHERE table_schema IN (:capstone_schema, 'bronze', 'silver', 'gold', 'metadata')
    ORDER BY table_schema, table_name
    """,
    params={"capstone_schema": CAPSTONE_SCHEMA},
)

display(sql_smoke_check_dataframe)

,table_schema,table_name
0,bronze,sensor_observations
1,capstone,bronze_observations_input_stage
2,capstone,data_quality_events
3,capstone,pipeline_artifacts
4,capstone,pipeline_runs
5,capstone,simulation_state_control
6,capstone,simulation_timing_config
7,capstone,synthetic_observations_premelt_stage
8,capstone,synthetic_observations_timestamped_stage
9,capstone,synthetic_pump_stream


---

## Start Logging for the Bronze Stage

Before doing the actual data work, I turn on logging for the Bronze stage. This gives the notebook a record of what happened during the run, including dataset resolution, ingest details, data shape, profiling results, warnings, and artifact outputs.

This matters because Bronze is the first formal layer of the pipeline. If something goes wrong later in Silver or Gold, I want to be able to trace back to what Bronze received, what it saved, and how it labeled the run.

In [9]:
'''
# Create bronze log path 
bronze_log_path = paths.logs / "bronze.log"

# Insure directory exists
bronze_log_path.parent.mkdir(parents=True, exist_ok=True)

# Initial Logger
capstone_logger = configure_logging(
    "capstone",
    bronze_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.bronze")

# Log load and initiation
logger.info("Bronze stage starting")

log_layer_paths(paths, current_layer="bronze", logger=logger)
'''

'\n# Create bronze log path \nbronze_log_path = paths.logs / "bronze.log"\n\n# Insure directory exists\nbronze_log_path.parent.mkdir(parents=True, exist_ok=True)\n\n# Initial Logger\ncapstone_logger = configure_logging(\n    "capstone",\n    bronze_log_path,\n    level=logging.DEBUG,\n    overwrite_handlers=True,\n)\n\n# Initiate Logger and log file\nlogger = logging.getLogger("capstone.bronze")\n\n# Log load and initiation\nlogger.info("Bronze stage starting")\n\nlog_layer_paths(paths, current_layer="bronze", logger=logger)\n'

----

In [10]:
log_layer_paths(paths, current_layer=CONTEXT_LAYER, logger=logger)

logger.info(
    "Project paths logged for %s layer",
    CONTEXT_LAYER,
    extra={"stage": CONTEXT_STAGE, "layer": CONTEXT_LAYER},
)

ledger.add(
    kind="step",
    step=f"{CONTEXT_LAYER}_paths_logged",
    message="Logged project layer paths.",
    data={
        "stage": CONTEXT_STAGE,
        "layer": CONTEXT_LAYER,
        "log_path": str(LOG_PATH),
    },
    logger=logger,
)


2026-06-14 20:36:46,498 | INFO | capstone.bronze | Project Root Path Loaded: /workspace
2026-06-14 20:36:46,500 | INFO | capstone.bronze | Project Logging Path Loaded: /workspace/logs
2026-06-14 20:36:46,502 | INFO | capstone.bronze | Project Artifacts Path Loaded: /workspace/artifacts
2026-06-14 20:36:46,505 | INFO | capstone.bronze | Project Notebooks Path Loaded: /workspace/notebooks
2026-06-14 20:36:46,508 | INFO | capstone.bronze | Project Truths Path Loaded: /workspace/artifacts/truths
2026-06-14 20:36:46,509 | INFO | capstone.bronze | Project Data Path Loaded: /workspace/data
2026-06-14 20:36:46,511 | INFO | capstone.bronze | Data Bronze Path Loaded: /workspace/data/bronze
2026-06-14 20:36:46,512 | INFO | capstone.bronze | Data Bronze Training Path Loaded: /workspace/data/bronze/train
2026-06-14 20:36:46,513 | INFO | capstone.bronze | Data Bronze Testing Path Loaded: /workspace/data/bronze/test
2026-06-14 20:36:46,515 | INFO | capstone.bronze | Project paths logged for bronze la

{'ts_utc': '2026-06-14T20:36:46.516793+00:00',
 'stage': 'bronze',
 'recipe': 'bronze__v001',
 'kind': 'step',
 'step': 'bronze_paths_logged',
 'message': 'Logged project layer paths.',
 'why': None,
 'consequence': None,
 'data': {'stage': 'bronze',
  'layer': 'bronze',
  'log_path': '/workspace/logs/bronze.log'}}

---

## Define Dataset Name Resolution Logic

This function handles the early dataset-name decision before the raw data is ingested. I need a dataset name early because some runtime tracking and setup steps need an identity before the dataframe exists.

However, this early value is only provisional. It should not control final artifact folder creation because the dataset itself may provide stronger evidence after ingest. This avoids the problem where a `PROVISIONAL_DATASET_NAME` creates folders or snapshots before the real dataset identity has been confirmed.

In [11]:
def resolve_dataset_name_for_bronze_pre_ingest(
    *,
    argument_value: Optional[str] = None,
    config_value: Optional[str] = None,
    handoff_dataset_name: Optional[str] = None,
    source_table_name: Optional[str] = None,
    source_table_dataset_map: Optional[dict] = None,
    fallback_value: Optional[str] = None,
    source_path: Optional[str] = None,
) -> Tuple[str, str, str]:
    """
    Resolve dataset name before Bronze ingestion.

    Priority order:
    1. CLI / Argument
    2. Config File
    3. Explicit handoff dataset name
    4. Source table -> dataset mapping
    5. Deterministic file-details-based generated name
    6. Fallback
    """

    def _normalize_dataset_name(dataset_name: str) -> str:
        normalized_value = str(dataset_name).strip().lower()
        normalized_value = normalized_value.replace(" ", "_")
        normalized_value = normalized_value.replace("-", "_")

        cleaned_characters = []
        for character in normalized_value:
            if character.isalnum() or character == "_":
                cleaned_characters.append(character)

        normalized_value = "".join(cleaned_characters)

        while "__" in normalized_value:
            normalized_value = normalized_value.replace("__", "_")

        normalized_value = normalized_value.strip("_")

        if normalized_value == "":
            raise ValueError("Dataset name normalization produced an empty value.")

        return normalized_value

    def _generate_deterministic_dataset_name_from_file_details(
        path_value: Optional[str],
    ) -> Optional[str]:
        if path_value is None or str(path_value).strip() == "":
            return None

        path_object = Path(path_value)

        file_stem_raw = path_object.stem.strip()
        if file_stem_raw == "":
            file_stem_raw = "dataset"

        file_stem_normalized = _normalize_dataset_name(file_stem_raw)

        file_size_bytes = "na"
        modified_timestamp = "na"
        content_fingerprint = "nohash"

        if path_object.exists() and path_object.is_file():
            stat_result = path_object.stat()
            file_size_bytes = str(int(stat_result.st_size))
            modified_timestamp = str(int(stat_result.st_mtime))

            try:
                sample_hasher = hashlib.sha1()

                with open(path_object, "rb") as file_handle:
                    first_chunk = file_handle.read(65536)
                    sample_hasher.update(first_chunk)

                    if stat_result.st_size > 65536:
                        seek_position = max(stat_result.st_size - 65536, 0)
                        file_handle.seek(seek_position)
                        last_chunk = file_handle.read(65536)
                        sample_hasher.update(last_chunk)

                sample_hasher.update(file_size_bytes.encode("utf-8"))
                sample_hasher.update(modified_timestamp.encode("utf-8"))

                content_fingerprint = sample_hasher.hexdigest()[:8]

            except Exception:
                content_fingerprint = "readfail"

        generated_dataset_name = (
            f"{file_stem_normalized}_{file_size_bytes}_{modified_timestamp}_{content_fingerprint}"
        )

        return _normalize_dataset_name(generated_dataset_name)

    source_table_dataset_map = source_table_dataset_map or {}

    if argument_value is not None and str(argument_value).strip() != "":
        return (
            _normalize_dataset_name(str(argument_value)),
            "argument",
            "argument",
        )

    if config_value is not None and str(config_value).strip() != "":
        return (
            _normalize_dataset_name(str(config_value)),
            "config",
            "config",
        )

    if handoff_dataset_name is not None and str(handoff_dataset_name).strip() != "":
        return (
            _normalize_dataset_name(str(handoff_dataset_name)),
            "handoff_dataset_name",
            "handoff",
        )

    if source_table_name is not None and str(source_table_name).strip() != "":
        mapped_dataset_name = source_table_dataset_map.get(str(source_table_name).strip())
        if mapped_dataset_name is not None and str(mapped_dataset_name).strip() != "":
            return (
                _normalize_dataset_name(str(mapped_dataset_name)),
                "source_table_name",
                "source_table_map",
            )

    generated_dataset_name = _generate_deterministic_dataset_name_from_file_details(source_path)

    if generated_dataset_name is not None:
        return (
            generated_dataset_name,
            "source_path",
            "file_details",
        )

    fallback_value_text = (
        fallback_value
        if (fallback_value is not None and str(fallback_value).strip() != "")
        else "unknown_dataset"
    )

    return (
        _normalize_dataset_name(str(fallback_value_text)),
        "fallback",
        "fallback",
    )

## Resolve the Provisional Dataset Identity

Here I call the dataset-name resolution logic to get an initial dataset identity before ingest. I am calling it provisional on purpose because this value may be updated once the raw data is loaded and inspected.

This step is useful for early runtime tracking, but it should not be treated as the final dataset name. The final Bronze artifact folder should be based on the resolved dataset name after ingestion, not on this provisional value.

In [12]:


PROVISIONAL_DATASET_NAME, PROVISIONAL_DATASET_SOURCE_COLUMN, PROVISIONAL_DATASET_METHOD = (
    resolve_dataset_name_for_bronze_pre_ingest(
        argument_value=DATASET_NAME_ARGUMENT,
        config_value=DATASET_NAME_CONFIG,
        handoff_dataset_name=(
            DATASET_NAME_POSTGRES if BRONZE_SOURCE_MODE == "postgres_handoff" else None
        ),
        source_table_name=(
            POSTGRES_SOURCE_TABLE_NAME if BRONZE_SOURCE_MODE == "postgres_handoff" else None
        ),
        source_table_dataset_map=POSTGRES_SOURCE_TABLE_DATASET_MAP,
        fallback_value="unknown_dataset",
        source_path=(
            str(RAW_FILE_PATH / RAW_FILE_NAME) if BRONZE_SOURCE_MODE == "file" else None
        ),
    )
)

## Prepare Dataset Resolution Metadata

This helper writes dataset-resolution metadata into the dataframe attributes after ingestion. That gives the notebook one clear place to store how the dataset name was resolved, what column or fallback was used, and which dataset identity should be treated as final.

The reason for doing this is to keep the naming logic traceable instead of guessing later. Bronze should be able to explain whether the dataset name came from an argument, config value, handoff field, source table, file name, or metadata column.

In [13]:
def write_dataset_resolution_attrs(
    dataframe,
    *,
    dataset_column: str = "meta__dataset",
    fallback_dataset_name: str | None = None,
    fallback_method: str = "fallback_dataset_name",
):
    """
    Write Bronze dataset resolution metadata into dataframe.attrs.

    Resolution order:
    1. Use dataset_column if it exists and contains exactly one non-null value
    2. Otherwise use fallback_dataset_name
    """

    resolved_dataset_name = None
    dataset_source_column = None
    dataset_method = None

    if dataset_column in dataframe.columns:
        dataset_values = (
            dataframe[dataset_column]
            .dropna()
            .astype(str)
            .str.strip()
        )

        unique_dataset_values = sorted(value for value in dataset_values.unique() if value)

        if len(unique_dataset_values) == 1:
            resolved_dataset_name = unique_dataset_values[0]
            dataset_source_column = dataset_column
            dataset_method = "dataset_column"

        elif len(unique_dataset_values) > 1:
            raise ValueError(
                f"Multiple dataset values found in '{dataset_column}': {unique_dataset_values}"
            )

    if resolved_dataset_name is None:
        if fallback_dataset_name is None or str(fallback_dataset_name).strip() == "":
            raise ValueError(
                "Could not resolve dataset name from dataframe column or fallback_dataset_name."
            )

        resolved_dataset_name = str(fallback_dataset_name).strip()
        dataset_source_column = None
        dataset_method = fallback_method

    dataframe.attrs["dataset_resolution"] = {
        "dataset_name": resolved_dataset_name,
        "dataset_source_column": dataset_source_column,
        "dataset_method": dataset_method,
    }

    return dataframe

## Defer the Config Snapshot Until the Final Dataset Name Exists

This cell intentionally does not save the config snapshot yet. The snapshot should be saved under the final resolved dataset name, not under the provisional name.

This is part of rolling back the unnecessary changes that made Bronze create outputs too early. The config snapshot still matters for repeatability and tracking, but it should be written after ingestion confirms `RESOLVED_DATASET_NAME`.

----

In [14]:
print("Config snapshot will be saved after final dataset resolution.")


Config snapshot will be saved after final dataset resolution.


----

## Initialize the Experiment Tracking Run

This cell starts the Weights & Biases run for the Bronze stage. The run tracks the source mode, dataset identity, input source, runtime settings, and other metadata that help explain what happened during this notebook execution.

At this point, the run may still use the provisional dataset name because the final dataframe has not been ingested yet. That is okay as long as the tracking metadata is updated later after `RESOLVED_DATASET_NAME` is confirmed.

In [15]:
if BRONZE_SOURCE_MODE == "file":
    WANDB_SOURCE_KIND = "file"
    WANDB_SOURCE_REFERENCE = str(RAW_FILE_PATH / RAW_FILE_NAME)
    WANDB_SOURCE_TABLE_NAME = None
    WANDB_RAW_DATA_FILE = RAW_FILE_NAME
    WANDB_RAW_PATH = str(RAW_FILE_PATH / RAW_FILE_NAME)

elif BRONZE_SOURCE_MODE == "postgres_handoff":
    WANDB_SOURCE_KIND = "postgres_handoff"
    WANDB_SOURCE_REFERENCE = f"postgres:public.{POSTGRES_SOURCE_TABLE_NAME}"
    WANDB_SOURCE_TABLE_NAME = POSTGRES_SOURCE_TABLE_NAME
    WANDB_RAW_DATA_FILE = None
    WANDB_RAW_PATH = None

else:
    raise ValueError(f"Unsupported BRONZE_SOURCE_MODE for W&B setup: {BRONZE_SOURCE_MODE}")

#### Start experiment tracking for this stage

This cell starts the Weights & Biases run for the notebook stage. The run metadata ties the executed notebook, dataset identity, and configuration profile to the saved artifacts.

In [16]:
wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="bronze",
    config={
        "dataset_name_provisional": PROVISIONAL_DATASET_NAME,
        "dataset_resolution_stage": "pre_ingest",
        "bronze_version": BRONZE_VERSION,

        "source_mode": BRONZE_SOURCE_MODE,
        "source_kind": WANDB_SOURCE_KIND,
        "source_reference": WANDB_SOURCE_REFERENCE,
        "source_table_name": WANDB_SOURCE_TABLE_NAME,

        "raw_data_file": WANDB_RAW_DATA_FILE,
        "raw_path": WANDB_RAW_PATH,

        "split_status": SPLIT_STATUS,
        #"label_type": LABEL_TYPE,
        #"label_source": LABEL_SOURCE,
        "run_id": RUN_ID,
        "asset_id": ASSET_ID,

        "add_record_id": ADD_RECORD_ID,
        "record_id_inputs": RECORD_ID_INPUTS if ADD_RECORD_ID else None,
        "record_id_method": RECORD_ID_METHOD if ADD_RECORD_ID else None,

        "bronze_out_path": str(BRONZE_DATA_PATH),
    },
)

logger.info("W&B initialized: %s", wandb_run.name)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: dcoo230 (dcoo230-western-governors-university). Use `wandb login --relogin` to force relogin


2026-06-14 20:37:08,560 | INFO | capstone.bronze | W&B initialized: bronze__001


----

## Ingest the Raw Dataset into the Bronze Layer

This is where the notebook actually loads the raw pump data into a dataframe. Bronze is focused on getting the raw dataset into a structured, traceable starting point for the rest of the pipeline.

The goal here is not to do heavy cleaning, feature engineering, or model preparation. The goal is to ingest the source data, preserve the raw signal as much as possible, attach the metadata needed for lineage, and prepare the dataframe for Bronze-level validation and output.

In [17]:
if BRONZE_SOURCE_MODE == "file":
    dataframe = ingest_data(
        RAW_FILE_PATH,
        file_name=RAW_FILE_NAME,
        dataset_name=DATASET_NAME_ARGUMENT,
        dataset_name_config=DATASET_NAME_CONFIG,
        dataset_candidates=DATASET_CANDIDATES,
        split=SPLIT_STATUS,
        run_id=RUN_ID,
        asset_id=ASSET_ID,
        add_record_id=True,
        validate=True,
    )

elif BRONZE_SOURCE_MODE == "postgres_handoff":
    engine = get_engine_from_env()

    dataframe = read_layer_dataframe(
        engine=engine,
        schema="public",
        table_name="bronze_observations_input_stage",
        where_clause="dataset_id = :dataset_id AND run_id = :run_id",
        params={
            "dataset_id": DATASET_NAME_POSTGRES,
            "run_id": RUN_ID,
        },
        order_by="batch_id, row_in_batch",
        require_exists=True,
    )

    dataframe = write_dataset_resolution_attrs(
        dataframe,
        dataset_column="meta__dataset",
        fallback_dataset_name=DATASET_NAME_POSTGRES,
        fallback_method="postgres_handoff",
    )

else:
    raise ValueError(f"Unsupported BRONZE_SOURCE_MODE: {BRONZE_SOURCE_MODE}")

2026-06-14 20:37:11,014 | INFO | capstone.file_io | Loading Data file: /workspace/data/raw/pump_sensor_data/sensor.csv
2026-06-14 20:37:14,922 | INFO | capstone.file_io | Loaded Data File: sensor.csv | shape=(220320, 55) | columns=['Unnamed: 0', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51', 'machine_status']
2026-06-14 20:37:14,927 | INFO | capstone.file_

----

## Confirm the Final Dataset Identity After Ingestion

After ingestion, I confirm what dataset identity was actually resolved. This matters because the dataframe may contain better evidence than the pre-ingest fallback value.

This is the point where the notebook moves from the provisional dataset name to the final resolved dataset name. From here forward, artifact folders, snapshots, lineage records, and tracking updates should use `RESOLVED_DATASET_NAME` so the outputs match the real Bronze dataset.

In [18]:
dataset_resolution = dataframe.attrs.get("dataset_resolution", {})

if not dataset_resolution:
    raise ValueError(
        "Bronze ingest did not write dataset_resolution metadata to dataframe.attrs."
    )

RESOLVED_DATASET_NAME = dataset_resolution.get("dataset_name")
DATASET_SOURCE_COLUMN = dataset_resolution.get("dataset_source_column")
DATASET_METHOD = dataset_resolution.get("dataset_method")

if RESOLVED_DATASET_NAME is None:
    raise ValueError("Bronze ingest did not return dataset_resolution metadata in dataframe.attrs.")

----

## Create the Bronze Artifact Folders with the Artifact Util

Now that the final dataset name has been resolved, I create the Bronze artifact directories using the project artifact utility.

This is the better method for Bronze because the artifact folders are created once, in one place, using `build_artifact_dirs()` and the final `RESOLVED_DATASET_NAME`. That avoids the earlier issue where the notebook used a `PROVISIONAL_DATASET_NAME` first and then resolved it later.

These folders organize the Bronze outputs into profiles, quality checks, schema records, summaries, metadata, config snapshots, and lineage files. This keeps the run outputs clean and makes it easier to audit what Bronze produced.

In [19]:

BRONZE_ARTIFACT_DIRS = build_artifact_dirs(
    artifacts_root=ARTIFACTS_ROOT,
    stage="bronze",
    dataset_name=RESOLVED_DATASET_NAME,
    family=None,
    subdirs=[
        "profiles",
        "quality",
        "schema",
        "summaries",
        "metadata",
        "config",
        "lineage",
    ],
)

BRONZE_ARTIFACTS_PATH = BRONZE_ARTIFACT_DIRS["root"]
BRONZE_PROFILE_DIR = BRONZE_ARTIFACT_DIRS["profiles"]
BRONZE_QUALITY_DIR = BRONZE_ARTIFACT_DIRS["quality"]
BRONZE_SCHEMA_DIR = BRONZE_ARTIFACT_DIRS["schema"]
BRONZE_SUMMARY_DIR = BRONZE_ARTIFACT_DIRS["summaries"]
BRONZE_METADATA_DIR = BRONZE_ARTIFACT_DIRS["metadata"]
BRONZE_CONFIG_DIR = BRONZE_ARTIFACT_DIRS["config"]
BRONZE_LINEAGE_DIR = BRONZE_ARTIFACT_DIRS["lineage"]

CONFIG_SNAPSHOT_PATH = (
    BRONZE_CONFIG_DIR
    / f"{RESOLVED_DATASET_NAME}__bronze__resolved_config.yaml"
)

BRONZE_ARTIFACT_SUMMARY_PATH = (
    BRONZE_SUMMARY_DIR
    / f"{RESOLVED_DATASET_NAME}__bronze__artifact_summary.json"
)

bronze_ledger_path = (
    BRONZE_LINEAGE_DIR
    / FILENAMES["bronze_ledger_file_name"]
)

if CONFIG["execution"].get("save_config_snapshot", True):
    export_config_snapshot(CONFIG, CONFIG_SNAPSHOT_PATH)

print("Bronze artifact root:", BRONZE_ARTIFACTS_PATH)
print("Config snapshot path:", CONFIG_SNAPSHOT_PATH)

BRONZE_ARTIFACT_DIRS


Bronze artifact root: /workspace/artifacts/bronze/pump
Config snapshot path: /workspace/artifacts/bronze/pump/config/pump__bronze__resolved_config.yaml


{'stage_dataset_root': PosixPath('/workspace/artifacts/bronze/pump'),
 'root': PosixPath('/workspace/artifacts/bronze/pump'),
 'profiles': PosixPath('/workspace/artifacts/bronze/pump/profiles'),
 'quality': PosixPath('/workspace/artifacts/bronze/pump/quality'),
 'schema': PosixPath('/workspace/artifacts/bronze/pump/schema'),
 'summaries': PosixPath('/workspace/artifacts/bronze/pump/summaries'),
 'metadata': PosixPath('/workspace/artifacts/bronze/pump/metadata'),
 'config': PosixPath('/workspace/artifacts/bronze/pump/config'),
 'lineage': PosixPath('/workspace/artifacts/bronze/pump/lineage')}

---

## Update Tracking Metadata with the Final Dataset Name

Now that the final dataset identity has been confirmed, I update the tracking run so it reflects the resolved values instead of only the provisional values.

This keeps the W&B run accurate. The provisional name is useful early in the notebook, but the final run metadata should show the dataset name that Bronze actually used for artifacts, lineage, and saved outputs.

In [20]:
wandb_run.config.update(
    {
        "dataset_name_final": RESOLVED_DATASET_NAME,
        "dataset_source_column": DATASET_SOURCE_COLUMN,
        "dataset_method": DATASET_METHOD,
        "dataset_resolution_stage": "bronze_ingest_final",
        "source_mode": BRONZE_SOURCE_MODE,
        "source_kind": WANDB_SOURCE_KIND,
        "source_reference": WANDB_SOURCE_REFERENCE,
        "source_table_name": WANDB_SOURCE_TABLE_NAME,
    },
    allow_val_change=True,
)

---

## Log Any Dataset Name Changes

This checkpoint records whether the dataset name changed between the pre-ingest provisional value and the post-ingest resolved value.

I want this logged because it explains why early setup metadata may not match the final artifact folder name. That is not necessarily an error; it just means the dataset identity was improved after the raw data became available.

In [21]:
if PROVISIONAL_DATASET_NAME != RESOLVED_DATASET_NAME:
    logger.info(
        "Dataset name changed after Bronze ingest when inside-dataset evidence became available | provisional=%s | resolved=%s",
        PROVISIONAL_DATASET_NAME,
        RESOLVED_DATASET_NAME,
    )

DATASET_NAME = RESOLVED_DATASET_NAME

----

## Build the Bronze Truth Record Foundation

Here I start the Bronze truth record. This is part of the lineage and audit structure for the project. It helps answer what data was used, where it came from, what stage processed it, and what metadata was attached.

For Bronze, this is especially important because it is the first medallion layer. Silver and Gold should be able to trace their inputs back to this Bronze output instead of relying on assumptions.

In [22]:
bronze_truth = initialize_layer_truth(
    truth_version=TRUTH_VERSION,
    dataset_name=DATASET_NAME,
    layer_name=LAYER_NAME,
    process_run_id=PROCESS_RUN_ID,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=None,
)

bronze_truth = update_truth_section(
    bronze_truth,
    "config_snapshot",
    {
        "bronze_version": BRONZE_VERSION,
        "split_status": SPLIT_STATUS,
        "label_type": LABEL_TYPE,
        "label_source": LABEL_SOURCE,
        "run_id": RUN_ID,
        "asset_id": ASSET_ID,
        "add_record_id": ADD_RECORD_ID,
        "record_id_inputs": RECORD_ID_INPUTS if ADD_RECORD_ID else None,
        "record_id_method": RECORD_ID_METHOD if ADD_RECORD_ID else None,
        "pipeline_mode": PIPELINE_MODE,
    },
)

bronze_truth = update_truth_section(
    bronze_truth,
    "runtime_facts",
    {
        "source_run_id": RUN_ID,
        "raw_file_path": str(RAW_FILE_PATH / RAW_FILE_NAME),
        "raw_data_dir": str(RAW_DATA_PATH),
        "dataset_name_provisional": PROVISIONAL_DATASET_NAME,
        "dataset_name_final": RESOLVED_DATASET_NAME,
        "dataset_source_column": DATASET_SOURCE_COLUMN,
        "dataset_method": DATASET_METHOD,
    },
)

bronze_truth = update_truth_section(
    bronze_truth,
    "artifact_paths",
    {
        "bronze_output_dir": str(BRONZE_DATA_PATH),
        "bronze_output_file_name": "pump__bronze__train.parquet",
    },
)

bronze_truth = update_truth_section(
    bronze_truth,
    "notes",
    {
        "purpose": "Bronze ingestion truth record",
    },
)

----

## Bronze Data Review

Before moving forward, I stop and look at the Bronze dataframe. At this stage, I am not doing deep feature engineering or final analysis yet. I am checking whether the ingest worked and whether the dataframe looks structurally usable.

This review checks shape, column names, data types, missingness, duplicate patterns, and basic descriptive information. The goal is to catch obvious issues before saving the Bronze layer or moving into Silver.

In [23]:
# Basic Dataframe Information/Summary

# Shape 
print("Shape:", dataframe.shape)
logger.info("Shape: %s", dataframe.shape)

# Dtypes as a compact block
print("\nData types:")
print(dataframe.dtypes)
logger.info("Dtypes:\n%s", dataframe.dtypes.to_string())

# Memory Usages
print("\nMemory usage (MB):")
print(dataframe.memory_usage(deep=True).sum() / (1024 ** 2))
mem_mb = dataframe.memory_usage(deep=True).sum() / (1024 ** 2)
logger.info("Memory usage (MB): %.2f", mem_mb)

# Head(15) as text (truncate columns for readability)
print("\nFirst 15 rows:")
display(dataframe.head(15))
logger.info("Head(15):\n%s", dataframe.head(15).to_string(max_cols=40, max_rows=15))

# Describe Numbers
print("\nBasic numeric summary:")
display(dataframe.describe(include=[np.number]).T)
desc_num = dataframe.describe(include=[np.number]).T
logger.info("Numeric describe (truncated):\n%s", desc_num.to_string(max_rows=60))

# Describe Objects and categorical
print("\nBasic object / categorical summary:")
object_category_columns = dataframe.select_dtypes(include=["object", "category", "string"]).columns
if len(object_category_columns):
    desc_obj = dataframe[object_category_columns].describe().T
    display(desc_obj)
    logger.info("Object/categorical describe (truncated):\n%s", desc_obj.to_string(max_rows=60))
else:
    logger.info("No object/category/string columns to describe.")

# Meta Columns
print("\nMeta Columns Summary:")
meta_columns = [column for column in dataframe.columns if column.startswith("meta__")]
dataframe[meta_columns].head(3)
logger.info("Meta Columns: (%d): %s", len(meta_columns), "\n".join(meta_columns))

# All Other Columns
print("\nAll Other Columns Summary:")
other_columns = [column for column in dataframe.columns if not column.startswith("meta__")]
dataframe[other_columns].head(3)
logger.info("All Other Columns: (%d): %s", len(other_columns), "\n".join(other_columns))

# Missing
missing = (dataframe[other_columns].isna().mean() * 100).sort_values(ascending=False).head(20)
display(missing.to_frame("pct_missing"))
logger.info("Top missingness (%%):\n%s", missing.to_string())


2026-06-14 20:37:26,198 | INFO | capstone.bronze | Shape: (220320, 63)
2026-06-14 20:37:26,209 | INFO | capstone.bronze | Dtypes:
meta__dataset                       category
meta__split                         category
meta__run_id                          object
meta__asset_id                        object
meta__ingested_at_utc    datetime64[us, UTC]
meta__source_file             string[python]
meta__source_row_id                    int64
meta__record_id                       uint64
Unnamed: 0                             int64
timestamp                             object
sensor_00                            float64
sensor_01                            float64
sensor_02                            float64
sensor_03                            float64
sensor_04                            float64
sensor_05                            float64
sensor_06                            float64
sensor_07                            float64
sensor_08                            float64
sensor_09      

Shape: (220320, 63)

Data types:
meta__dataset                       category
meta__split                         category
meta__run_id                          object
meta__asset_id                        object
meta__ingested_at_utc    datetime64[us, UTC]
                                ...         
sensor_48                            float64
sensor_49                            float64
sensor_50                            float64
sensor_51                            float64
machine_status                        object
Length: 63, dtype: object

Memory usage (MB):


2026-06-14 20:37:26,605 | INFO | capstone.bronze | Memory usage (MB): 165.63


165.62513065338135

First 15 rows:


,meta__dataset,meta__split,meta__run_id,meta__asset_id,meta__ingested_at_utc,meta__source_file,meta__source_row_id,meta__record_id,Unnamed: 0,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,machine_status
0,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,0,14598431322315673869,0,2018-04-01 00:00:00,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,NaN,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.989580,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
1,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,1,15954729095895098000,1,2018-04-01 00:01:00,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,NaN,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.989580,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
2,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,2,10041703297090838359,2,2018-04-01 00:02:00,2.444734,47.35243,53.2118,46.397570,638.8889,73.54598,13.32465,16.03733,15.61777,15.01013,37.86777,48.17723,32.08894,1.708474,420.8480,NaN,462.7798,459.6364,2.500062,666.2234,399.9418,880.4237,501.3617,982.7342,631.1326,740.8031,849.8997,454.2390,778.5734,715.6266,661.5740,721.8750,694.7721,441.2635,169.9820,343.1955,200.9694,93.90508,41.40625,31.25000,69.53125,30.468750,31.770830,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,241.3194,203.7037,NORMAL
3,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,3,2072036352569063261,3,2018-04-01 00:03:00,2.460474,47.09201,53.1684,46.397568,628.1250,76.98898,13.31742,16.24711,15.69734,15.08247,38.57977,48.65607,31.67221,1.579427,420.7494,NaN,462.8980,460.8858,2.509521,666.0114,399.1046,878.8917,499.0430,977.7520,625.4076,739.2722,847.7579,474.8731,779.5091,690.4011,686.1111,754.6875,683.3831,446.2493,166.4987,343.9586,193.1689,101.04060,41.92708,31.51042,72.13541,30.468750,31.510420,40.88541,39.062500,64.81481,51.21528,38.194440,155.9606,66.84028,240.4514,203.1250,NORMAL
4,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,4,4365040424004714369,4,2018-04-01 00:04:00,2.445718,47.13541,53.2118,46.397568,636.4583,76.58897,13.35359,16.21094,15.69734,15.08247,39.48939,49.06298,31.95202,1.683831,419.8926,NaN,461.4906,468.2206,2.604785,663.2111,400.5426,882.5874,498.5383,979.5755,627.1830,737.6033,846.9182,408.8159,785.2307,704.6937,631.4814,766.1458,702.4431,433.9081,164.7498,339.9630,193.8770,101.70380,42.70833,31.51042,76.82291,30.989580,31.510420,41.40625,38.773150,65.10416,51.79398,38.773150,158.2755,66.55093,242.1875,201.3889,NORMAL
5,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,5,13731207034657248969,5,2018-04-01 00:05:00,2.453588,47.09201,53.1684,46.397568,637.6157,78.18568,13.41146,16.16753,15.89265,15.16204,39.29406,49.37051,32.23816,1.673484,418.9049,NaN,461.8948,461.9289,2.507935,663.4962,398.6428,872.4973,498.40

...

,count,unique,top,freq
meta__dataset,220320,1,pump,220320
meta__split,220320,1,unsplit,220320
meta__run_id,220320,1,run__001,220320
meta__asset_id,220320,1,asset__001,220320
meta__source_file,220320,1,sensor.csv,220320
timestamp,220320,220320,2018-08-31 23:43:00,1
machine_status,220320,3,NORMAL,205836


2026-06-14 20:37:28,272 | INFO | capstone.bronze | Object/categorical describe (truncated):
                    count  unique                  top    freq
meta__dataset      220320       1                 pump  220320
meta__split        220320       1              unsplit  220320
meta__run_id       220320       1             run__001  220320
meta__asset_id     220320       1           asset__001  220320
meta__source_file  220320       1           sensor.csv  220320
timestamp          220320  220320  2018-08-31 23:43:00       1
machine_status     220320       3               NORMAL  205836
2026-06-14 20:37:28,289 | INFO | capstone.bronze | Meta Columns: (8): meta__dataset
meta__split
meta__run_id
meta__asset_id
meta__ingested_at_utc
meta__source_file
meta__source_row_id
meta__record_id
2026-06-14 20:37:28,339 | INFO | capstone.bronze | All Other Columns: (55): Unnamed: 0
timestamp
sensor_00
sensor_01
sensor_02
sensor_03
sensor_04
sensor_05
sensor_06
sensor_07
sensor_08
sensor_09
sensor_


Meta Columns Summary:

All Other Columns Summary:


,pct_missing
sensor_15,100.000000
sensor_50,34.956881
sensor_51,6.982117
sensor_00,4.633261
sensor_07,2.474129
sensor_08,2.317992
sensor_06,2.177741
sensor_09,2.085603
sensor_01,0.167484
sensor_30,0.118464


2026-06-14 20:37:28,419 | INFO | capstone.bronze | Top missingness (%):
sensor_15    100.000000
sensor_50     34.956881
sensor_51      6.982117
sensor_00      4.633261
sensor_07      2.474129
sensor_08      2.317992
sensor_06      2.177741
sensor_09      2.085603
sensor_01      0.167484
sensor_30      0.118464
sensor_29      0.032680
sensor_32      0.030864
sensor_18      0.020879
sensor_17      0.020879
sensor_22      0.018609
sensor_25      0.016340
sensor_16      0.014070
sensor_45      0.012255
sensor_39      0.012255
sensor_43      0.012255


----

### Ask

What does this Bronze review tell me, and is the dataset in a good enough state to move forward?

### Answer

This check gives me a first-pass structural read on the ingested dataset. I am looking for problems that would make the Bronze output unreliable, such as missing key columns, unexpected data types, obvious duplicate rows, major missingness, or a dataset shape that does not match expectations.

If this review looks reasonable, then the dataset is in a good enough state to save as Bronze and continue into the later medallion layers. If something looks wrong here, I should fix the ingest or source handling before pushing the problem downstream.

---

## Finalize Lineage Metadata and Save the Bronze Truth Record

Now I am turning the Bronze dataframe into a traceable pipeline output. This step fingerprints the source, identifies metadata and feature columns, updates the truth record, and writes lineage information into the dataframe.

The reason for this is simple: I want the Bronze output to be explainable later. If Silver or Gold uses this dataset, I should be able to trace the data back to the raw source, the Bronze run, the pipeline mode, and the truth hash created at this step.

In [24]:
bronze_source_fingerprint = build_file_fingerprint(RAW_FILE_PATH / RAW_FILE_NAME)
bronze_truth = update_truth_section(
    bronze_truth,
    "source_fingerprint",
    bronze_source_fingerprint,
)

bronze_meta_columns = sorted(
    set(
        identify_meta_columns(dataframe)
        + [
            "meta__truth_hash",
            "meta__parent_truth_hash",
            "meta__pipeline_mode",
        ]
    )
)

bronze_feature_columns = identify_feature_columns(dataframe)

bronze_truth_record = build_truth_record(
    truth_base=bronze_truth,
    row_count=len(dataframe),
    column_count=dataframe.shape[1] + 3,
    meta_columns=bronze_meta_columns,
    feature_columns=bronze_feature_columns,
)

BRONZE_TRUTH_HASH = bronze_truth_record["truth_hash"]

dataframe = stamp_truth_columns(
    dataframe,
    truth_hash=BRONZE_TRUTH_HASH,
    parent_truth_hash=None,
    pipeline_mode=PIPELINE_MODE,
)

bronze_truth_path = save_truth_record(
    bronze_truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name=LAYER_NAME,
)

append_truth_index(
    bronze_truth_record,
    truth_index_path=TRUTH_INDEX_PATH,
)

logger.info("Bronze truth hash: %s", BRONZE_TRUTH_HASH)
logger.info("Bronze truth path: %s", bronze_truth_path)
logger.info("Bronze process_run_id: %s", PROCESS_RUN_ID)

print("Bronze truth hash:", BRONZE_TRUTH_HASH)
print("Bronze truth path:", bronze_truth_path)
print("Bronze process_run_id:", PROCESS_RUN_ID)

2026-06-14 20:37:34,827 | INFO | capstone.bronze | Bronze truth hash: b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69
2026-06-14 20:37:34,829 | INFO | capstone.bronze | Bronze truth path: /workspace/artifacts/truths/bronze/pump__bronze__truth__b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69.json
2026-06-14 20:37:34,832 | INFO | capstone.bronze | Bronze process_run_id: bronze_process__20260614T203612Z


Bronze truth hash: b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69
Bronze truth path: /workspace/artifacts/truths/bronze/pump__bronze__truth__b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69.json
Bronze process_run_id: bronze_process__20260614T203612Z


----

## Define a Helper to Reorder Bronze Columns

This helper keeps the dataframe layout consistent by moving metadata columns to the front. I do this mostly for clarity and standardization.

The values are not being changed here. This is only about making the Bronze output easier to inspect, easier to validate, and easier for later notebooks to consume.

In [25]:
def collect_meta_columns(existing_columns: List[str]) -> List[str]:
    meta_columns: List[str] = []
    for column in existing_columns:
        if column.startswith("meta__"):
            meta_columns.append(column)
    return meta_columns



def reorder_bronze_columns(dataframe: pd.DataFrame) -> pd.DataFrame:
    existing_columns = list(dataframe.columns)

    meta_columns = collect_meta_columns(existing_columns)

    bronze_columns: List[str] = []
    for column in existing_columns:
        if column not in meta_columns:
            bronze_columns.append(column)

    final_order: List[str] = []
    final_order.extend(meta_columns)
    final_order.extend(bronze_columns)

    return dataframe[final_order].copy()

## Reorder the Bronze Dataframe Columns

Here I apply the column-ordering helper so the metadata fields come first and the original feature fields follow after them.

This makes the Bronze dataframe easier to read because the lineage fields are visible at the front. It also helps keep the project consistent without changing the actual sensor values or analytical content of the dataset.

In [26]:
# Reorder dataframe and put meta columns in the front. 

dataframe = reorder_bronze_columns(dataframe)

meta_columns_after_reorder = collect_meta_columns(list(dataframe.columns))

non_meta_columns_after_reorder = [
    column_name
    for column_name in dataframe.columns
    if not column_name.startswith("meta__")
]

logger.info(
    "Bronze columns reordered successfully. "
    "Meta columns moved to the front while preserving original within-group order. "
    "meta_column_count=%d | non_meta_column_count=%d | total_column_count=%d",
    len(meta_columns_after_reorder),
    len(non_meta_columns_after_reorder),
    dataframe.shape[1],
)


2026-06-14 20:37:40,030 | INFO | capstone.bronze | Bronze columns reordered successfully. Meta columns moved to the front while preserving original within-group order. meta_column_count=11 | non_meta_column_count=55 | total_column_count=66


----

## Save the Bronze Dataset

At this point the Bronze dataframe has been ingested, reviewed, stamped with lineage metadata, and reordered into a clean structure. Now I save it as the Bronze layer output.

This saved file becomes the stable handoff point for Silver. Bronze should preserve the raw data structure as much as possible while adding the metadata needed for traceability and downstream processing.

In [27]:
# Save Data as Parquet
save_data(dataframe, paths.data_bronze_train, BRONZE_TRAIN_DATA_FILE_NAME)

2026-06-14 20:37:42,648 | INFO | capstone.file_io | Saving DataFrame to Parquet: /workspace/data/bronze/train/pump__bronze__train.parquet
2026-06-14 20:37:44,489 | INFO | capstone.file_io | Saved: pump__bronze__train.parquet | shape=(220320, 66) | columns=['meta__dataset', 'meta__split', 'meta__run_id', 'meta__asset_id', 'meta__ingested_at_utc', 'meta__source_file', 'meta__source_row_id', 'meta__record_id', 'meta__truth_hash', 'meta__parent_truth_hash', 'meta__pipeline_mode', 'Unnamed: 0', 'timestamp', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37'

PosixPath('/workspace/data/bronze/train/pump__bronze__train.parquet')

----

## Create Bronze Profiling Outputs

This step generates profiling and diagnostic outputs for the Bronze dataframe. The purpose is to preserve a record of the dataset’s condition at the Bronze stage.

These outputs help document column types, missingness, summary statistics, and other basic quality information. That gives me evidence for what Bronze looked like before any Silver cleaning or transformation happens.

In [28]:
metrics, saved = profile_dataframe(
    dataframe,
    logger,
    artifacts_dir=BRONZE_PROFILE_DIR,
)

2026-06-14 20:37:47,727 | INFO | capstone.bronze | Shape: (220320, 66)
2026-06-14 20:37:47,730 | INFO | capstone.bronze | Memory (MB): 209.12
2026-06-14 20:37:47,734 | INFO | capstone.bronze | Dtypes:
meta__dataset                         category
meta__split                           category
meta__run_id                            object
meta__asset_id                          object
meta__ingested_at_utc      datetime64[us, UTC]
meta__source_file               string[python]
meta__source_row_id                      int64
meta__record_id                         uint64
meta__truth_hash                        object
meta__parent_truth_hash                 object
meta__pipeline_mode                     object
Unnamed: 0                               int64
timestamp                               object
sensor_00                              float64
sensor_01                              float64
sensor_02                              float64
sensor_03                              float64


----

## Finalize Bronze Run Tracking

Now I wrap up the Bronze tracking outputs by sending the final dataset, logs, notebook reference, and artifact directories into the run finalization step.

This makes the notebook easier to audit because the run has a clear record of what was produced. It also keeps the project organized by connecting the Bronze dataframe, profiling outputs, config snapshot, logs, and lineage artifacts to the same tracked run.

In [29]:
finalize_wandb_stage(
    wandb_run,
    stage="bronze",
    dataframe=dataframe,
    project_root=paths.root,
    logs_dir=paths.logs,
    dataset_dirs=[paths.data_bronze_train],
    dataset_artifact_name=BRONZE_TRAIN_DATA_FILE_NAME,
    notebook_path=paths.notebooks / "Preprocessing" / "EDA_Notebook_Pump_Bronze_01_Preprocessing.ipynb",
    profile=False,
)


{'metrics': {'rows': 220320, 'cols': 66, 'memory_mb': 209.11863040924072},
 'saved': {},
 'artifacts_stage_dir': PosixPath('/workspace/artifacts/bronze'),
 'stage_log': PosixPath('/workspace/logs/bronze.log'),
 'dataset_dirs': [PosixPath('/workspace/data/bronze/train')]}

----

## Finish the Tracking Run

This cell closes the Weights & Biases run after the Bronze outputs have been finalized. Closing the run cleanly helps make sure the tracked metadata, files, and run status are saved correctly.

This is a small step, but it keeps the notebook from leaving the tracking session open after Bronze has completed.

In [30]:
wandb_run.finish()

cols,▁
memory_mb,▁
rows,▁
cols,66
memory_mb,209.11863
rows,220320


----

## Define Validation Helpers for Bronze Checks

These helper functions make the final validation checks stricter and easier to read. They help make sure required values exist and have the expected types before the notebook reports the Bronze stage as complete.

This is not a new coding standard; it is just a safety layer around the final checks so errors are easier to understand if something is missing or malformed.

In [31]:
def require_dict(value: Any | None, name: str) -> Dict[str, Any]:
    if value is None:
        raise ValueError(f"{name} cannot be None.")

    if not isinstance(value, dict):
        raise TypeError(
            f"{name} must be a dictionary, got {type(value).__name__}: {value!r}"
        )

    return cast(Dict[str, Any], value)

#### Define integer validation helper

This cell defines helper logic used by the surrounding `Define Validation Helpers for Bronze Checks` section. Keeping the helper directly before its use makes the notebook assumptions visible and easier to audit.

In [32]:
def require_int_value(value: object, name: str) -> int:
    if value is None:
        raise ValueError(f"{name} cannot be None.")

    try:
        return int(cast(Any, value))
    except (TypeError, ValueError) as exc:
        raise TypeError(
            f"{name} must be convertible to int, "
            f"got {type(value).__name__}: {value!r}"
        ) from exc

---

## Validate Bronze Lineage and Truth Consistency

Before I consider the Bronze stage complete, I run a sanity check on the lineage fields and saved truth record.

This is basically a trust check. I want to confirm that the dataframe contains the expected metadata columns, the truth hash is present, and the saved truth information lines up with the Bronze output. If this passes, I can be more confident that the Bronze output is not just saved, but traceable.

In [33]:
required_bronze_meta_columns = [
    "meta__truth_hash",
    "meta__parent_truth_hash",
    "meta__pipeline_mode",
]

missing_bronze_meta_columns = [
    column_name
    for column_name in required_bronze_meta_columns
    if column_name not in dataframe.columns
]
if missing_bronze_meta_columns:
    raise ValueError(
        f"Bronze dataframe is missing required lineage columns: {missing_bronze_meta_columns}"
    )

bronze_dataframe_truth_hash = extract_truth_hash(dataframe)
if bronze_dataframe_truth_hash is None:
    raise ValueError("Bronze dataframe does not contain a readable meta__truth_hash value.")

if bronze_dataframe_truth_hash != BRONZE_TRUTH_HASH:
    raise ValueError(
        "Bronze dataframe truth hash does not match BRONZE_TRUTH_HASH:\n"
        f"dataframe={bronze_dataframe_truth_hash}\n"
        f"record={BRONZE_TRUTH_HASH}"
    )

bronze_parent_values = (
    dataframe["meta__parent_truth_hash"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

if bronze_parent_values:
    raise ValueError(
        "Bronze should not have a populated parent truth hash, but found values:\n"
        f"{bronze_parent_values}"
    )

if not Path(bronze_truth_path).exists():
    raise FileNotFoundError(f"Bronze truth file was not created: {bronze_truth_path}")

loaded_bronze_truth_raw = load_json(bronze_truth_path)

loaded_bronze_truth = require_dict(
    loaded_bronze_truth_raw,
    "loaded_bronze_truth",
)

if "truth_hash" not in loaded_bronze_truth:
    raise KeyError("Saved Bronze truth file is missing required key: truth_hash")

loaded_bronze_truth_hash = str(loaded_bronze_truth["truth_hash"]).strip()

if loaded_bronze_truth_hash != BRONZE_TRUTH_HASH:
    raise ValueError(
        "Saved Bronze truth file hash does not match BRONZE_TRUTH_HASH:\n"
        f"file={loaded_bronze_truth_hash}\n"
        f"record={BRONZE_TRUTH_HASH}"
    )

parent_truth_hash = loaded_bronze_truth.get("parent_truth_hash")

if parent_truth_hash is not None:
    raise ValueError(
        "Bronze truth file parent_truth_hash should be None.\n"
        f"Found: {parent_truth_hash}"
    )

if "row_count" not in loaded_bronze_truth:
    raise KeyError("Saved Bronze truth file is missing required key: row_count")

if "column_count" not in loaded_bronze_truth:
    raise KeyError("Saved Bronze truth file is missing required key: column_count")

bronze_truth_row_count = require_int_value(
    loaded_bronze_truth["row_count"],
    "loaded_bronze_truth['row_count']",
)

bronze_truth_column_count = require_int_value(
    loaded_bronze_truth["column_count"],
    "loaded_bronze_truth['column_count']",
)

dataframe_row_count = int(len(dataframe))
dataframe_column_count = int(dataframe.shape[1])

if bronze_truth_row_count != dataframe_row_count:
    raise ValueError(
        "Bronze truth row_count does not match dataframe row count:\n"
        f"truth={bronze_truth_row_count}\n"
        f"dataframe={dataframe_row_count}"
    )

if bronze_truth_column_count != dataframe_column_count:
    raise ValueError(
        "Bronze truth column_count does not match stamped dataframe column count:\n"
        f"truth={bronze_truth_column_count}\n"
        f"dataframe={dataframe_column_count}"
    )

print("Bronze lineage sanity check passed.")

2026-06-14 20:38:05,766 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/truths/bronze/pump__bronze__truth__b90b025534b790481484fe4870e5bff95c50143212898e9ff02b88401303da69.json


Bronze lineage sanity check passed.


----

## Final Bronze Structural Check

I am doing one more quick review of the dataframe before moving on from the Bronze notebook. This is another lightweight inspection step.

The earlier review checked the data after ingest. This final review confirms that the dataframe still looks structurally sound after lineage stamping, column reordering, profiling, and saving. This gives me a final checkpoint before SQL write and downstream handoff.

In [34]:
print("=== Bronze: Data Overview ===")
print(f"Shape: {dataframe.shape[0]:,} rows × {dataframe.shape[1]} columns\n")

print("=== Column Types ===")
display(dataframe.dtypes.to_frame("dtype").head(20))

print("\n=== df.info() ===")
display(dataframe.info())

print("\n=== Basic Descriptive Statistics (numeric only) ===")
display(dataframe.describe().T)

print("\n=== Top Sample Rows ===")
display(dataframe.head())


=== Bronze: Data Overview ===
Shape: 220,320 rows × 66 columns

=== Column Types ===


,dtype
meta__dataset,category
meta__split,category
meta__run_id,object
meta__asset_id,object
meta__ingested_at_utc,"datetime64[us, UTC]"
meta__source_file,string[python]
meta__source_row_id,int64
meta__record_id,uint64
meta__truth_hash,object
meta__parent_truth_hash,object



=== df.info() ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220320 entries, 0 to 220319
Data columns (total 66 columns):
 #   Column                   Non-Null Count   Dtype              
---  ------                   --------------   -----              
 0   meta__dataset            220320 non-null  category           
 1   meta__split              220320 non-null  category           
 2   meta__run_id             220320 non-null  object             
 3   meta__asset_id           220320 non-null  object             
 4   meta__ingested_at_utc    220320 non-null  datetime64[us, UTC]
 5   meta__source_file        220320 non-null  string             
 6   meta__source_row_id      220320 non-null  int64              
 7   meta__record_id          220320 non-null  uint64             
 8   meta__truth_hash         220320 non-null  object             
 9   meta__parent_truth_hash  0 non-null       object             
 10  meta__pipeline_mode      220320 non-null  object             

None


=== Basic Descriptive Statistics (numeric only) ===


,count,mean,std,min,25%,50%,75%,max
meta__source_row_id,220320.0,1.101595e+05,6.360105e+04,0.000000e+00,5.507975e+04,1.101595e+05,1.652392e+05,2.203190e+05
meta__record_id,220320.0,9.224243e+18,5.322367e+18,1.873763e+14,4.610703e+18,9.220552e+18,1.381850e+19,1.844674e+19
Unnamed: 0,220320.0,1.101595e+05,6.360105e+04,0.000000e+00,5.507975e+04,1.101595e+05,1.652392e+05,2.203190e+05
sensor_00,210112.0,2.372221e+00,4.122274e-01,0.000000e+00,2.438831e+00,2.456539e+00,2.499826e+00,2.549016e+00
sensor_01,219951.0,4.759161e+01,3.296666e+00,0.000000e+00,4.631076e+01,4.813368e+01,4.947916e+01,5.672743e+01
sensor_02,220301.0,5.086739e+01,3.666820e+00,3.315972e+01,5.039062e+01,5.164930e+01,5.277777e+01,5.603299e+01
sensor_03,220301.0,4.375248e+01,2.418887e+00,3.164062e+01,4.283854e+01,4.422743e+01,4.531250e+01,4.822049e+01
sensor_04,220301.0,5.906739e+02,1.440239e+02,2.798032e+00,6.266204e+02,6.326389e+02,6.376157e+02,8.000000e+02
sensor_05,220301.0,7.339641e+01,1.729825e+01,0.000000e+00,6.997626e+01,7.557679e+01,8.091215e+01,9.999988e+01
sensor_06,215522.0,1.350154e+01,2.163736e+00,1.446759e-02,1.334635e+01,1.364294e+01,1.453993e+01,2.225116e+01



=== Top Sample Rows ===


,meta__dataset,meta__split,meta__run_id,meta__asset_id,meta__ingested_at_utc,meta__source_file,meta__source_row_id,meta__record_id,meta__truth_hash,meta__parent_truth_hash,meta__pipeline_mode,Unnamed: 0,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_50,sensor_51,machine_status
0,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,0,14598431322315673869,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,0,2018-04-01 00:00:00,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,NaN,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
1,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,1,15954729095895098000,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,1,2018-04-01 00:01:00,2.465394,47.09201,53.2118,46.310760,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,NaN,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,243.0556,201.3889,NORMAL
2,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,2,10041703297090838359,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,2,2018-04-01 00:02:00,2.444734,47.35243,53.2118,46.397570,638.8889,73.54598,13.32465,16.03733,15.61777,15.01013,37.86777,48.17723,32.08894,1.708474,420.8480,NaN,462.7798,459.6364,2.500062,666.2234,399.9418,880.4237,501.3617,982.7342,631.1326,740.8031,849.8997,454.2390,778.5734,715.6266,661.5740,721.8750,694.7721,441.2635,169.9820,343.1955,200.9694,93.90508,41.40625,31.25000,69.53125,30.46875,31.770830,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,241.3194,203.7037,NORMAL
3,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,3,2072036352569063261,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,3,2018-04-01 00:03:00,2.460474,47.09201,53.1684,46.397568,628.1250,76.98898,13.31742,16.24711,15.69734,15.08247,38.57977,48.65607,31.67221,1.579427,420.7494,NaN,462.8980,460.8858,2.509521,666.0114,399.1046,878.8917,499.0430,977.7520,625.4076,739.2722,847.7579,474.8731,779.5091,690.4011,686.1111,754.6875,683.3831,446.2493,166.4987,343.9586,193.1689,101.04060,41.92708,31.51042,72.13541,30.46875,31.510420,40.88541,39.062500,64.81481,51.21528,38.194440,155.9606,66.84028,240.4514,203.1250,NORMAL
4,pump,unsplit,run__001,asset__001,2026-06-14 20:37:14.939771+00:00,sensor.csv,4,4365040424004714369,b90b025534b790481484fe4870e5bff95c50143212898e...,None,batch,4,2018-04-01 00:04:00,2.445718,47.13541,53.2118,46.397568,636.4583,76.58897,13.35359,16.21094,15.69734,15.08247,39.48939,49.06298,31.95202,1.683831,419.8926,NaN,461.4906,468.2206,2.604785,663.2111,400.5426,882.5874,498.5383,979.5755,627.1830,737.6033,846.9182,408.8159,785.2307,704.6937,631.4814,766.1458,702.4431,433.9081,164.7498,339.9630,193.8770,101.70380,42.70833,31.51042,76.82291,30.98958,31.510420,41.40625,38.773150,65.10416,51.79398,38.773150,158.2

### Ask

What does this final review confirm about the Bronze output?

### Answer

This final check confirms that the Bronze dataframe still has the expected structure after the notebook has added metadata, saved outputs, and completed profiling. I am checking that the row and column counts still make sense, that metadata columns are present, and that the dataframe is ready to act as the Bronze handoff for later stages.

This is not meant to prove the dataset is fully cleaned or model-ready. It confirms that Bronze did its job: ingest the raw data, preserve traceability, create artifacts, and save a stable starting point for Silver.

----

## Bronze SQL Write Cell

Target: `bronze.sensor_observations`

This step writes the final Bronze dataframe into Postgres as row-level JSON payloads while preserving dataset/run lineage.

The purpose is to support the database side of the capstone in addition to the file-based medallion outputs. The parquet output gives the notebook pipeline a stable handoff, while the SQL write gives the operational/database layer a queryable Bronze table.

In [35]:

WRITE_TO_POSTGRES = True

if WRITE_TO_POSTGRES:
    
    bronze_sql_summary_dataframe = write_bronze_sensor_observations_sql(
        engine=engine,
        capstone_schema=CAPSTONE_SCHEMA,
        layer_schema=LAYER_SCHEMA,
        dataset_id=DATASET_ID,
        run_id=RUN_ID,
        dataframe=dataframe,
        notebook_globals=globals(),
        dataset_name=globals().get("DATASET_NAME", DATASET_ID),
    )

    display(bronze_sql_summary_dataframe)

else:
    print("Postgres write skipped.")

Using supplied dataframe -> 220,320 rows x 66 columns
Deleted 0 existing rows from bronze.sensor_observations.
Wrote 220,320 rows.
Upserted pipeline run metadata: run__001__bronze_preprocessing
Logged DQ event: bronze.sensor_observations | bronze_sql_write | pass


,dataset_id,run_id,row_count
0,pump,run__001,220320


## Preview the Bronze SQL Output

After writing the Bronze dataframe to Postgres, I preview the target table to confirm the write worked. This is a quick database-side check that verifies rows exist in `bronze.sensor_observations` and that the SQL layer can read them back.

This does not replace the dataframe checks or the saved artifact checks. It simply confirms that the final Bronze output also made it into the database layer successfully.

In [36]:
read_sql_dataframe(
    engine,
    """
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema = :schema
      AND table_name = 'sensor_observations'
    """,
    params={"schema": LAYER_SCHEMA},
)

,table_schema,table_name
0,bronze,sensor_observations


#### Preview the SQL-facing layer output

This cell writes, previews, or closes out a SQL-facing layer output. Reconcile the row count from this step against the saved file artifact, ledger entry, and Postgres validation query.

In [37]:
preview_sql_table(
    schema=LAYER_SCHEMA,
    table="sensor_observations",
    limit=5,
)

,bronze_id,dataset_id,run_id,asset_id,event_time,event_step,time_index,source_table,source_row_id,raw_payload,meta_truth_hash,meta_parent_truth_hash,meta_ingested_at_utc
0,1,pump,run__001,asset__001,2018-04-01 04:00:00+00:00,0,0,sensor.csv,14598431322315673869,"{'index': 0, 'sensor_00': 2.465394, 'sensor_01...",b90b025534b790481484fe4870e5bff95c50143212898e...,None,2026-06-14 20:40:36.244786+00:00
1,2,pump,run__001,asset__001,2018-04-01 04:01:00+00:00,1,1,sensor.csv,15954729095895098000,"{'index': 1, 'sensor_00': 2.465394, 'sensor_01...",b90b025534b790481484fe4870e5bff95c50143212898e...,None,2026-06-14 20:40:36.244786+00:00
2,3,pump,run__001,asset__001,2018-04-01 04:02:00+00:00,2,2,sensor.csv,10041703297090838359,"{'index': 2, 'sensor_00': 2.444734, 'sensor_01...",b90b025534b790481484fe4870e5bff95c50143212898e...,None,2026-06-14 20:40:36.244786+00:00
3,4,pump,run__001,asset__001,2018-04-01 04:03:00+00:00,3,3,sensor.csv,2072036352569063261,"{'index': 3, 'sensor_00': 2.460474, 'sensor_01...",b90b025534b790481484fe4870e5bff95c50143212898e...,None,2026-06-14 20:40:36.244786+00:00
4,5,pump,run__001,asset__001,2018-04-01 04:04:00+00:00,4,4,sensor.csv,4365040424004714369,"{'index': 4, 'sensor_00': 2.445718, 'sensor_01...",b90b025534b790481484fe4870e5bff95c50143212898e...,None,2026-06-14 20:40:36.244786+00:00


#### Close the database engine cleanly

This cell performs the next transformation in `Preview the Bronze SQL Output`. Review the output or logged message before relying on this result downstream.

In [38]:
# Dispose Engine
engine.dispose()

---

## Summary

This notebook preserves the current analytical behavior while documenting the role of the Bronze step in the capstone workflow. The code cells above should continue to produce the same configured outputs, artifacts, logger messages, and ledger entries as before this documentation pass.


## Next Stage

Silver Pre-EDA uses the Bronze output as the cleaned raw foundation for profiling and metadata checks.
